# 4. ResStock aggregation to transformer nodes

**Purpose:** build residential transformer-node load profiles (`<xfr_gisid>_RES`) from the ResStock buildings matched to each residential AMI meter, for one or more ResStock scenarios (baseline, upgrades, demand flexibility). Residential meters without a ResStock match keep their measured AMI load (AMI fallback); they are kept out of the ResStock total, flagged and reported. Every output file name carries the scenario name.

**Position in the pipeline:** last step; needs notebook 0 (section 3), notebook 3 and `results/match_report.csv`. See `docs/pipeline_summary.md`.

| # | Section | Key output (in `results/`) |
|---|---------|-----------|
| 1 | Parameters | — |
| 2 | Load meter map, match report and AMI fallback data | — |
| 3 | Meter match status | `resstock_meter_status.csv` |
| 4 | Aggregate each scenario | `aggregated_load_nodes_wide_resstock_RES_<scenario>.*` (ResStock only), `aggregated_load_nodes_wide_ami_fallback_RES_<scenario>_ami-<gap_tag>.*` (AMI of unmatched meters), `aggregated_load_nodes_wide_resstock_plus_ami_fallback_RES_<scenario>_ami-<gap_tag>.*` (sum), `resstock_aggregation_report_<scenario>.csv` (per transformer), `resstock_meter_sources_<scenario>.csv` (per meter) |
| 5 | Scenario summary | — |

**Inputs**
- `data/parquet/resstock_timeseries_all_15_<scenario>.parquet` from `3_resstock_load_table_build.ipynb`. The special name `"baseline"` reads `resstock_timeseries_all_15.parquet`, the original (not rebuilt) baseline run of notebook 1.
- `results/match_report.csv`: `meter id` → `ResStock_ids`.
- `results/ami_meter_xfr_map.csv` and `data/parquet/ami_meter_wide_15_<gap_tag>.parquet` from section 3 of `0_ami_data_prep_and_aggregation.ipynb` (run it first with the gap method you want).

**Rules**
- Residential (`RES`) meters only. Non-residential columns (`_NRS`) come from the AMI aggregation of notebook 0.
- A meter matched to *k* ResStock buildings gets the average of the *k* series. The same building matched to several meters counts once per meter (the old notebook de-duplicated buildings per transformer and under-counted).
- A meter on several transformers is split equally (`1 / n_xfr`), as in notebook 0.
- Unmatched residential meters (empty match list, or not in the match report) → AMI fallback. Meters that also have no AMI contribute nothing and are flagged `no_data`.
- Matched buildings missing from a scenario file (upgrade not applicable to the building): `MISSING_BLDG_POLICY = "baseline"` uses the building's `BASELINE_SCENARIO` series; `"ami"` sends the meter to the AMI fallback.

**Time alignment caveats** (all series are kWh per 15 min, interval-end)
- ResStock: the EnergyPlus `24:00` value of each day was relabelled as `00:00` of the *same* day when the load tables were built (notebooks 1 and 3), so 1 of 96 intervals per day is 24 h early. `FIX_EPLUS_MIDNIGHT = True` moves those rows to the next day (output names get a `_midfix` suffix). The first interval of the year (Jan 1 00:15) is 0 in the ResStock output.
- ResStock is in local standard time; AMI `ENDTIME_EST` is local clock time with daylight saving (fall-back hour averaged, spring-forward hour missing).
- The simulations use a 2023 calendar (Jan 1 = Sunday) with the 2024 weather mapped by month and day; 2024 starts on a Monday, so the EnergyPlus weekday is one day behind the AMI weekday before Feb 29 and two days behind after it. Weekday/weekend patterns therefore line up only approximately.


## Setup
Assumes the project environment is already created and activated (`python -m pip install -r requirements.txt`). The two setup cells below are the same in every notebook of this project; run them first. Run notebooks from `grid_edge_fy25/notebooks/` (all paths are relative to it).

In [1]:
# Preflight check: assumes the environment was created from requirements.txt
import importlib.util

required_modules = ['pandas', 'numpy', 'pyarrow']
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(
        "Missing Python packages: " + ", ".join(missing) +
        "\nCreate/activate the project environment and run: python -m pip install -r requirements.txt"
    )
print("Preflight OK: environment is ready.")

Preflight OK: environment is ready.


In [2]:
# Common setup (identical in all notebooks of this project)
import ast
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Paths (relative to grid_edge_fy25/notebooks/)
current_path = Path(os.getcwd())
data_path = current_path / ".." / "data"
base_files = data_path / "base_files"        # building id lists, measures, weather, workflow
input_files = data_path / "input_files"      # NY metadata, AMI consumption
parquet_folder = data_path / "parquet"       # load tables written for the optimization
results_path = current_path / ".." / "results"
extract_dir = data_path / "resstock_models"  # bldgXXXXXXX folders (models + simulation output)

parquet_folder.mkdir(parents=True, exist_ok=True)

---
## 1. Parameters
- `SCENARIOS`: suffixes of `data/parquet/resstock_timeseries_all_15_<scenario>.parquet` (same names as `SCENARIOS` in notebook 3); `"baseline"` reads `resstock_timeseries_all_15.parquet` (original notebook 1 run).
- `BASELINE_SCENARIO` / `MISSING_BLDG_POLICY`: what to do with matched buildings that are absent from a scenario file.
- `AMI_GAP_TAG`: must match the `GAP_TAG` printed by notebook 0 section 3 (`"zero"`, `"nan"`, `"interp4"`, ...).
- `FIX_EPLUS_MIDNIGHT`: see the time alignment caveats above.


In [ ]:
SCENARIOS = ["baseline_rebuilt", "up04_rebuilt", "up04_df_2F"]
BASELINE_SCENARIO = "baseline_rebuilt"   # used when MISSING_BLDG_POLICY == "baseline"
MISSING_BLDG_POLICY = "baseline"         # "baseline" | "ami"
AMI_GAP_TAG = "zero"                     # gap tag of the notebook 0 outputs to use
FIX_EPLUS_MIDNIGHT = False               # True: move ResStock 00:00 rows to the next day
MATCH_REPORT = results_path / "match_report.csv"


def resstock_file(scenario):
    """15-min ResStock table of a scenario; "baseline" = original notebook 1 run (no suffix)."""
    name = "resstock_timeseries_all_15" + ("" if scenario == "baseline" else f"_{scenario}")
    return parquet_folder / f"{name}.parquet"


if MISSING_BLDG_POLICY not in {"baseline", "ami"}:
    raise ValueError(f"Unknown MISSING_BLDG_POLICY: {MISSING_BLDG_POLICY}")
needed_files = [resstock_file(s) for s in SCENARIOS]
if MISSING_BLDG_POLICY == "baseline":
    needed_files.append(resstock_file(BASELINE_SCENARIO))
needed_files += [MATCH_REPORT,
                 results_path / "ami_meter_xfr_map.csv",
                 parquet_folder / f"ami_meter_wide_15_{AMI_GAP_TAG}.parquet"]
missing_files = [str(f) for f in needed_files if not f.exists()]
if missing_files:
    raise FileNotFoundError("Missing inputs:\n" + "\n".join(missing_files))

# Full 2024 15-min grid (interval-end), Feb 29 removed -> 35,040 timestamps
time_index = pd.date_range("2024-01-01 00:00", "2024-12-31 23:45", freq="15min", name="time")
time_index = time_index[time_index.strftime("%m-%d") != "02-29"]
print("Scenarios:", SCENARIOS, "| AMI gap tag:", AMI_GAP_TAG, "| midnight fix:", FIX_EPLUS_MIDNIGHT)


Scenarios: ['baseline_rebuilt', 'up04_rebuilt', 'up04_df_2F'] | AMI gap tag: zero | midnight fix: False


---
## 2. Load meter map, match report and AMI fallback data
Residential rows of the notebook 0 meter → transformer map, the meter-wide AMI table (residential meters only, already gap-handled), and the match report (`ResStock_ids` is a string list such as `"[13761]"` or `"[]"`).


In [4]:
meter_xfr = pd.read_csv(results_path / "ami_meter_xfr_map.csv",
                        dtype={"servicepointid": str, "xfr_gisid": str})
res_map = meter_xfr[meter_xfr["sector"] == "RES"].reset_index(drop=True)
res_columns = sorted(res_map["column"].unique())

ami_wide = pd.read_parquet(parquet_folder / f"ami_meter_wide_15_{AMI_GAP_TAG}.parquet")
if not ami_wide.index.equals(time_index):
    raise ValueError("AMI meter-wide table is not on the 2024 15-min grid without Feb 29")
ami_wide.index = time_index
ami_wide = ami_wide[[c for c in ami_wide.columns if c in set(res_map["servicepointid"])]]

mr = pd.read_csv(MATCH_REPORT, usecols=["meter id", "ResStock_ids"])
mr["servicepointid"] = mr["meter id"].astype(str)
if mr["servicepointid"].duplicated().any():
    raise ValueError("Duplicate meter ids in the match report")
mr["bldg_ids"] = mr["ResStock_ids"].apply(
    lambda s: [int(b) for b in ast.literal_eval(s)] if isinstance(s, str) else [])
match = mr.set_index("servicepointid")["bldg_ids"]

ignored = sorted(set(match.index) - set(res_map["servicepointid"]))
print("Residential meters on a transformer:", res_map["servicepointid"].nunique(),
      "| RES columns:", len(res_columns), "| with AMI:", ami_wide.shape[1])
print("Match report meters:", len(match), "| not a residential meter on a transformer (ignored):", ignored)

Residential meters on a transformer: 565 | RES columns: 146 | with AMI: 560
Match report meters: 538 | not a residential meter on a transformer (ignored): []


---
## 3. Meter match status
One row per residential meter: `match_status` = `matched` (≥1 ResStock ID), `empty_match` (in the report with an empty list) or `not_in_report`; `has_ami` from notebook 0. Saved to `results/resstock_meter_status.csv`.


In [5]:
meters = (res_map.drop_duplicates("servicepointid")
                 .set_index("servicepointid")[["n_xfr", "has_ami"]])
in_report = meters.index.isin(match.index)
meters["bldg_ids"] = [match.get(s, []) for s in meters.index]
meters["n_bldgs"] = meters["bldg_ids"].str.len()
meters["match_status"] = np.where(~in_report, "not_in_report",
                                  np.where(meters["n_bldgs"] > 0, "matched", "empty_match"))

meters.assign(bldg_ids=meters["bldg_ids"].astype(str)).to_csv(results_path / "resstock_meter_status.csv")

needed_bldgs = sorted({b for ids in meters["bldg_ids"] for b in ids})
print("Unique matched ResStock buildings:", len(needed_bldgs))
print("Meters with >1 matched building (averaged):", int((meters["n_bldgs"] > 1).sum()))
pd.crosstab(meters["match_status"], meters["has_ami"], margins=True)

Unique matched ResStock buildings: 187
Meters with >1 matched building (averaged): 0


has_ami,False,True,All
match_status,,,
empty_match,0,116,116
matched,0,422,422
not_in_report,5,22,27
All,5,560,565


---
## 4. Aggregate each scenario
For each scenario:
1. Load the matched buildings as a wide table (time × bldg_id) and check the time grid; apply `MISSING_BLDG_POLICY` to buildings absent from the scenario.
2. Assign each meter a source: `resstock` (matched and all its buildings available), `ami_fallback` (otherwise, if it has AMI) or `no_data`.
3. ResStock part per column = Σ over `resstock` meters of `weight / k` × building series; AMI part = Σ over `ami_fallback` meters of `weight` × meter AMI (missing values propagate, as in notebook 0).
4. Save the three wide tables (CSV + parquet, index `time`), the per-transformer report and the per-meter source table, and check energy totals.


In [6]:
def load_resstock_wide(scenario, bldg_ids):
    """Wide ResStock table (time x bldg_id) on the 2024 grid for the requested buildings present in the file."""
    df = pd.read_parquet(resstock_file(scenario), columns=["bldg_id", "time", "ts_elec"],
                         filters=[("bldg_id", "in", list(bldg_ids))])
    if FIX_EPLUS_MIDNIGHT:
        # 00:00 rows hold the 24:00 value of the same date -> move to the next day (no Feb 29; Dec 31 wraps to Jan 1)
        t = df["time"]
        midnight = (t.dt.hour == 0) & (t.dt.minute == 0)
        shifted = t[midnight] + pd.Timedelta(days=1)
        shifted = shifted.where(shifted.dt.strftime("%m-%d") != "02-29", shifted + pd.Timedelta(days=1))
        shifted = shifted.where(shifted.dt.year == 2024, shifted - pd.DateOffset(years=1))
        df.loc[midnight, "time"] = shifted
    wide = df.pivot(index="time", columns="bldg_id", values="ts_elec")
    if len(wide.index.symmetric_difference(time_index)) or wide.isna().any().any():
        raise ValueError(f"{scenario}: ResStock series are not complete on the 2024 15-min grid without Feb 29")
    wide = wide.reindex(time_index)
    wide.columns.name = None
    return wide


base_wide = (load_resstock_wide(BASELINE_SCENARIO, needed_bldgs)
             if MISSING_BLDG_POLICY == "baseline" else None)
midfix = "_midfix" if FIX_EPLUS_MIDNIGHT else ""
summary_rows = []

for scenario in SCENARIOS:
    tag = f"{scenario}{midfix}"
    print(f"\n=== {tag} ===")

    # 1. ResStock buildings for this scenario (+ baseline fill)
    rs_wide = load_resstock_wide(scenario, needed_bldgs)
    absent = [b for b in needed_bldgs if b not in rs_wide.columns]
    filled = []
    if base_wide is not None:
        filled = [b for b in absent if b in base_wide.columns]
        rs_wide = pd.concat([rs_wide, base_wide[filled]], axis=1)
    available = set(rs_wide.columns)
    print(f"Matched buildings: {len(needed_bldgs)} | in scenario: {len(needed_bldgs) - len(absent)} "
          f"| filled from {BASELINE_SCENARIO}: {len(filled)} | unavailable: {len(absent) - len(filled)}")

    # 2. Source per meter
    src = meters.copy()
    src["missing_bldgs"] = [[b for b in ids if b not in available] for ids in src["bldg_ids"]]
    src["baseline_filled_bldgs"] = [[b for b in ids if b in filled] for ids in src["bldg_ids"]]
    use_rs = (src["match_status"] == "matched") & (src["missing_bldgs"].str.len() == 0)
    src["source"] = np.where(use_rs, "resstock", np.where(src["has_ami"], "ami_fallback", "no_data"))
    src["fallback_reason"] = np.where(
        use_rs, "",
        np.where(src["match_status"] == "matched", "bldg_missing_in_scenario", src["match_status"]))

    rows = res_map.merge(src[["source", "bldg_ids", "n_bldgs", "baseline_filled_bldgs"]],
                         left_on="servicepointid", right_index=True)

    # 3a. ResStock part: weight / k per (meter, building); repeated buildings count once per meter
    rs_rows = rows[rows["source"] == "resstock"].explode("bldg_ids")
    rs_rows = rs_rows.assign(bldg_id=rs_rows["bldg_ids"].astype("int64"),
                             w=rs_rows["weight"] / rs_rows["n_bldgs"])
    agg_rs = pd.DataFrame(0.0, index=time_index, columns=res_columns)
    for col, g in rs_rows.groupby("column"):
        agg_rs[col] = rs_wide[g["bldg_id"].tolist()].to_numpy() @ g["w"].to_numpy()

    # 3b. AMI fallback part
    fb_rows = rows[rows["source"] == "ami_fallback"]
    agg_fb = pd.DataFrame(0.0, index=time_index, columns=res_columns)
    for col, g in fb_rows.groupby("column"):
        agg_fb[col] = (ami_wide[g["servicepointid"].tolist()]
                       .mul(g["weight"].to_numpy(), axis=1)
                       .sum(axis=1, skipna=False))

    agg_total = agg_rs + agg_fb

    # 4. Save
    outputs = {
        f"aggregated_load_nodes_wide_resstock_RES_{tag}": agg_rs,
        f"aggregated_load_nodes_wide_ami_fallback_RES_{tag}_ami-{AMI_GAP_TAG}": agg_fb,
        f"aggregated_load_nodes_wide_resstock_plus_ami_fallback_RES_{tag}_ami-{AMI_GAP_TAG}": agg_total,
    }
    for name, table in outputs.items():
        table.to_csv(results_path / f"{name}.csv")
        table.to_parquet(results_path / f"{name}.parquet")

    rows["baseline_fill"] = (rows["source"] == "resstock") & (rows["baseline_filled_bldgs"].str.len() > 0)
    report = rows.groupby("column").agg(
        xfr_gisid=("xfr_gisid", "first"),
        n_meters=("servicepointid", "size"),
        n_resstock=("source", lambda s: int((s == "resstock").sum())),
        n_ami_fallback=("source", lambda s: int((s == "ami_fallback").sum())),
        n_no_data=("source", lambda s: int((s == "no_data").sum())),
        n_baseline_fill=("baseline_fill", "sum"),
        n_split=("n_xfr", lambda s: int((s > 1).sum())),
    )
    report["kWh_resstock"] = agg_rs.sum()
    report["kWh_ami_fallback"] = agg_fb.sum()
    report["kWh_total"] = report["kWh_resstock"] + report["kWh_ami_fallback"]
    report["ami_fallback_share"] = report["kWh_ami_fallback"] / report["kWh_total"].replace(0, np.nan)
    report["nan_intervals"] = agg_total.isna().sum()
    report.reset_index().to_csv(results_path / f"resstock_aggregation_report_{tag}.csv", index=False)

    list_cols = ["bldg_ids", "missing_bldgs", "baseline_filled_bldgs"]
    src.assign(**{c: src[c].astype(str) for c in list_cols}).to_csv(
        results_path / f"resstock_meter_sources_{tag}.csv")

    # Energy checks
    expected_rs = float((rs_wide[rs_rows["bldg_id"].tolist()].sum().to_numpy() * rs_rows["w"].to_numpy()).sum())
    assert np.isclose(agg_rs.sum().sum(), expected_rs, rtol=1e-9), "ResStock energy not conserved"
    expected_fb = float((ami_wide[fb_rows["servicepointid"].tolist()].sum().to_numpy() * fb_rows["weight"].to_numpy()).sum())
    print(f"ResStock part: {agg_rs.sum().sum():,.0f} kWh (check OK) | AMI fallback: {agg_fb.sum().sum():,.0f} kWh "
          f"(meter total {expected_fb:,.0f}; differs only if fallback columns have NaN intervals)")

    counts = src["source"].value_counts()
    flagged = src[src["source"] != "resstock"]
    print("Meters by source:", counts.to_dict())
    print("Flagged meters by reason:", flagged.groupby(["source", "fallback_reason"]).size().to_dict())

    summary_rows.append({
        "scenario": tag,
        "meters_resstock": int(counts.get("resstock", 0)),
        "meters_ami_fallback": int(counts.get("ami_fallback", 0)),
        "meters_no_data": int(counts.get("no_data", 0)),
        "bldgs_filled_from_baseline": len(filled),
        "MWh_resstock": agg_rs.sum().sum() / 1000,
        "MWh_ami_fallback": agg_fb.sum().sum() / 1000,
        "MWh_total": agg_total.sum().sum() / 1000,
        "columns_with_nan": int((agg_total.isna().sum() > 0).sum()),
    })
    print("Saved:", *[f"{n}.csv/.parquet" for n in outputs], sep="\n  ")


=== baseline_rebuilt ===
Matched buildings: 187 | in scenario: 187 | filled from baseline_rebuilt: 0 | unavailable: 0
ResStock part: 3,399,426 kWh (check OK) | AMI fallback: 614,011 kWh (meter total 614,011; differs only if fallback columns have NaN intervals)
Meters by source: {'resstock': 422, 'ami_fallback': 138, 'no_data': 5}
Flagged meters by reason: {('ami_fallback', 'empty_match'): 116, ('ami_fallback', 'not_in_report'): 22, ('no_data', 'not_in_report'): 5}
Saved:
  aggregated_load_nodes_wide_resstock_RES_baseline_rebuilt.csv/.parquet
  aggregated_load_nodes_wide_ami_fallback_RES_baseline_rebuilt_ami-zero.csv/.parquet
  aggregated_load_nodes_wide_resstock_plus_ami_fallback_RES_baseline_rebuilt_ami-zero.csv/.parquet

=== up04_rebuilt ===
Matched buildings: 187 | in scenario: 109 | filled from baseline_rebuilt: 78 | unavailable: 0
ResStock part: 5,064,646 kWh (check OK) | AMI fallback: 614,011 kWh (meter total 614,011; differs only if fallback columns have NaN intervals)
Meters b

---
## 5. Scenario summary
Meter counts by source and annual energy (MWh) per scenario. `MWh_ami_fallback` is measured load of unmatched residential meters and is the same in every scenario (except when `MISSING_BLDG_POLICY = "ami"` moves extra meters to the fallback).

In [7]:
summary = pd.DataFrame(summary_rows).set_index("scenario")
summary.round(1)

,meters_resstock,meters_ami_fallback,meters_no_data,bldgs_filled_from_baseline,MWh_resstock,MWh_ami_fallback,MWh_total,columns_with_nan
scenario,,,,,,,,
baseline_rebuilt,422,138,5,0,3399.4,614.0,4013.4,0
up04_rebuilt,422,138,5,78,5064.6,614.0,5678.7,0
up04_df_2F,422,138,5,78,5015.4,614.0,5629.4,0
